In [35]:
import numpy as np
W_matrix=np.array([[1,-1,1,-1],[1,1,-1,-1],[-1,1,1,-1],[-1,-1,-1,1]])
v=np.array([[10,20,30,40]]).T
i=W_matrix.T@v
i_hand=(i.flatten().tolist())

In [27]:
%%writefile crossbar.sv
`timescale 1ns/1ps
module crossbar (
    input  wire signed [7:0] v0, v1, v2, v3,
    output reg  signed [9:0] i0, i1, i2, i3
);
    wire signed [9:0] s0 = v0;
    wire signed [9:0] s1 = v1;
    wire signed [9:0] s2 = v2;
    wire signed [9:0] s3 = v3;

    always @(*) begin
        i0 =  s0 + s1 - s2 - s3;
        i1 = -s0 + s1 + s2 - s3;
        i2 =  s0 - s1 + s2 - s3;
        i3 = -s0 - s1 - s2 + s3;
    end
endmodule



Overwriting crossbar.sv


In [8]:
import subprocess, re, numpy as np

def simulate_crossbar(v):
    v0, v1, v2, v3 = [int(x) for x in v]
    tb = f"""\
`timescale 1ns/1ps
module tb;
    reg  signed [7:0] v0, v1, v2, v3;
    wire signed [9:0] i0, i1, i2, i3;
    crossbar dut (.v0(v0),.v1(v1),.v2(v2),.v3(v3),
                  .i0(i0),.i1(i1),.i2(i2),.i3(i3));
    initial begin
        v0={v0}; v1={v1}; v2={v2}; v3={v3};
        #1;
        $display("i0=%0d i1=%0d i2=%0d i3=%0d",
                 $signed(i0),$signed(i1),$signed(i2),$signed(i3));
        $finish;
    end
endmodule
"""
    with open('tb_crossbar.sv','w') as f: f.write(tb)
    r = subprocess.run(['iverilog','-g2012','-o','sim_crossbar',
                        'tb_crossbar.sv','crossbar.sv'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("Compile error:\n", r.stderr); return None
    r = subprocess.run(['vvp','sim_crossbar'], capture_output=True, text=True)
    m = re.search(r'i0=(-?\d+) i1=(-?\d+) i2=(-?\d+) i3=(-?\d+)', r.stdout)
    return [int(m.group(k)) for k in range(1,5)] if m else None

In [37]:
v = [10, 20, 30, 40]
hw  = simulate_crossbar(v)
print(i_hand)
print(hw)
i_hand==hw

[-40, 0, -20, -20]
[-40, 0, -20, -20]


True

In [43]:
import numpy as np

W       = np.array([[1,-1,1,-1],[1,1,-1,-1],[-1,1,1,-1],[-1,-1,-1,1]])
R       = 1.0
R_sense = 0.001

def measure_all_columns(v):
    """Ground each column in turn, return measured currents and floating voltages."""
    v = np.array(v, dtype=float)
    results = {}
    for c in range(4):
        I = (W[:, c] * v).sum() / R / (1 + R_sense * np.abs(W[:, c]).sum() / R)
        floats = {}
        for cf in range(4):
            if cf == c: continue
            SW  = W[:, cf].sum()
            SWV = (W[:, cf] * v).sum()
            if   abs(SW)  > 1e-10:  floats[cf] = f"{SWV/SW:+.2f}V"
            elif abs(SWV) < 1e-10:  floats[cf] = "0V"
            else:                   floats[cf] = "RAILS"
        results[c] = {'I': I, 'floats': floats}
    return results

print("Unit impulse test — one column grounded at a time")
print("="*75)

for k in range(4):
    v_unit = [0]*4
    v_unit[k] = 1
    v_unit = np.array(v_unit, dtype=float)
    i_ideal = W.T @ v_unit
    res = measure_all_columns(v_unit)

    print(f"\nInput: v{k}=1, rest=0")
    print(f"  {'Col':>4}  {'I_sense':>9}  {'I_ideal':>9}  {'Error':>9}  Floating voltages")
    print(f"  {'-'*65}")
    for c in range(4):
        I      = res[c]['I']
        floats = res[c]['floats']
        fstr   = "  ".join(f"V_col{cf}={v}" for cf,v in floats.items())
        print(f"  {c:>4}  {I:>9.4f}  {i_ideal[c]:>9.4f}  {I-i_ideal[c]:>9.6f}  {fstr}")

Unit impulse test — one column grounded at a time

Input: v0=1, rest=0
   Col    I_sense    I_ideal      Error  Floating voltages
  -----------------------------------------------------------------
     0     0.9960     1.0000  -0.003984  V_col1=RAILS  V_col2=RAILS  V_col3=+0.50V
     1    -0.9960    -1.0000   0.003984  V_col0=RAILS  V_col2=RAILS  V_col3=+0.50V
     2     0.9960     1.0000  -0.003984  V_col0=RAILS  V_col1=RAILS  V_col3=+0.50V
     3    -0.9960    -1.0000   0.003984  V_col0=RAILS  V_col1=RAILS  V_col2=RAILS

Input: v1=1, rest=0
   Col    I_sense    I_ideal      Error  Floating voltages
  -----------------------------------------------------------------
     0     0.9960     1.0000  -0.003984  V_col1=RAILS  V_col2=RAILS  V_col3=+0.50V
     1     0.9960     1.0000  -0.003984  V_col0=RAILS  V_col2=RAILS  V_col3=+0.50V
     2    -0.9960    -1.0000   0.003984  V_col0=RAILS  V_col1=RAILS  V_col3=+0.50V
     3    -0.9960    -1.0000   0.003984  V_col0=RAILS  V_col1=RAILS  V_col

The unit impulse table is the weight matrix W.T itself — each row shows exactly which columns a given input drives, and the small error is just the I × R_sense voltage drop across the sense resistor. The RAILS entries tell you which floating columns have no DC operating point when that input is active.

In [11]:
import numpy as np

W = np.array([[1,-1,1,-1],[1,1,-1,-1],[-1,1,1,-1],[-1,-1,-1,1]])
v = np.array([10, 20, 30, 40], dtype=float)

R       = 1.0
R_sense = 0.001      # very small sense resistor

i_ideal = W.T @ v

print(f"{'Col':>4} {'I_sense':>10} {'I_ideal':>10} {'Error':>10}  Floating column voltages")
print("-"*70)

for c in range(4):
    # Solve: I = Σ W[r,c]*(v[r] - I*R_sense)/R
    I = (W[:, c] * v).sum() / R / (1 + R_sense * np.abs(W[:, c]).sum() / R)

    floats = {}
    for cf in range(4):
        if cf == c: continue
        SW  = W[:, cf].sum()           # signed weight sum
        SWV = (W[:, cf] * v).sum()    # weighted voltage sum
        if   abs(SW)  > 1e-10:  floats[f"V_col{cf}"] = f"{SWV/SW:+.2f}V"
        elif abs(SWV) < 1e-10:  floats[f"V_col{cf}"] = "0V (balanced)"
        else:                   floats[f"V_col{cf}"] = "RAILS"

    print(f"  {c:>2}  {I:>10.4f}  {i_ideal[c]:>10.4f}  {I-i_ideal[c]:>10.6f}  {floats}")

 Col    I_sense    I_ideal      Error  Floating column voltages
----------------------------------------------------------------------
   0    -39.8406    -40.0000    0.159363  {'V_col1': '0V (balanced)', 'V_col2': 'RAILS', 'V_col3': '+10.00V'}
   1      0.0000      0.0000    0.000000  {'V_col0': 'RAILS', 'V_col2': 'RAILS', 'V_col3': '+10.00V'}
   2    -19.9203    -20.0000    0.079681  {'V_col0': 'RAILS', 'V_col1': '0V (balanced)', 'V_col3': '+10.00V'}
   3    -19.9203    -20.0000    0.079681  {'V_col0': 'RAILS', 'V_col1': '0V (balanced)', 'V_col2': 'RAILS'}


Three things stand out:

Error ≈ −R_sense × I_ideal — the tiny sense resistor causes a small voltage drop, slightly reducing the measured current. Make R_sense smaller and the error vanishes

Column 3 always floats at +10V regardless of which column is being measured — that's the unbalanced column settling to its KCL voltage

Columns 0 and 2 show RAILS — their signed weight sum is zero but their ideal current is non-zero, so KCL has no DC solution and in a real circuit those nodes would charge up to the supply rail. This is where the sneak current problem actually lives

Looking at the full vector result: v=[10,20,30,40] gives i1 = -10+20+30-40 = 0.
Zero output ≠ zero activity. Here's what's actually happening at that node:

| Cell in row 1 | Weight | Input (V) | Cell current (mA) |
|---------------|--------|-----------|-------------------|
| (1,0)         |   -1   |    10     |        -10        |
| (1,1)         |   +1   |    20     |        +20        |
| (1,2)         |   +1   |    30     |        +30        |
| (1,3)         |   -1   |    40     |        -40        |
| **Sum**       |        |           |        **0**      |


For row 1, the individual sneak currents are all non-zero — they just cancel exactly at the output node. Physically this means:The sense resistor reads 0V — indistinguishable from "nothing happened". Power is still dissipated in every cell in that row. This is a decision boundary case — if this output feeds a classifier, i1=0 means the sample sits exactly on the hyperplane between two classes (maximally ambiguous). The danger in a real analog crossbar is noise. At exact cancellation the SNR is worst — any device mismatch, thermal noise, or IR drop shifts the result off zero, and you can't tell which side is correct. A digital crossbar like this one gives you exact zero, but a memristor-based one would give you a noisy near-zero that could flip sign unpredictably.

## Split Differential Sensing (SDS) Mitigation

Instead of one bipolar sense per column, `crossbar_sds` routes positive-weight rows and negative-weight rows onto **separate sub-buses**, each with its own sense resistor. The output is `I[c] = I_c+ − I_c−`.

Key changes vs the original:
- Each sub-bus carries only unipolar current → no floating-node / RAILS condition
- R_sense errors partially cancel in the differential subtraction
- At the zero crossing (col 1 with v=[10,20,30,40]): `I_c+ = I_c- ≈ 50` — both sub-measurements are large and well-conditioned even though their difference is 0

In [1]:
%%writefile crossbar_sds.sv
`timescale 1ns/1ps
module crossbar_sds (
    input  wire signed [7:0] v0, v1, v2, v3,
    output wire signed [9:0] i0, i1, i2, i3
);
    wire signed [9:0] s0 = v0;
    wire signed [9:0] s1 = v1;
    wire signed [9:0] s2 = v2;
    wire signed [9:0] s3 = v3;

    // col 0: W[:,0]=[+1,+1,-1,-1]  ->  i0 = (v0+v1) - (v2+v3)
    wire signed [9:0] i0_pos = s0 + s1;
    wire signed [9:0] i0_neg = s2 + s3;
    assign i0 = i0_pos - i0_neg;

    // col 1: W[:,1]=[-1,+1,+1,-1]  ->  i1 = (v1+v2) - (v0+v3)
    wire signed [9:0] i1_pos = s1 + s2;
    wire signed [9:0] i1_neg = s0 + s3;
    assign i1 = i1_pos - i1_neg;

    // col 2: W[:,2]=[+1,-1,+1,-1]  ->  i2 = (v0+v2) - (v1+v3)
    wire signed [9:0] i2_pos = s0 + s2;
    wire signed [9:0] i2_neg = s1 + s3;
    assign i2 = i2_pos - i2_neg;

    // col 3: W[:,3]=[-1,-1,-1,+1]  ->  i3 = v3 - (v0+v1+v2)
    wire signed [9:0] i3_pos = s3;
    wire signed [9:0] i3_neg = s0 + s1 + s2;
    assign i3 = i3_pos - i3_neg;

endmodule


Overwriting crossbar_sds.sv


In [9]:
import subprocess, re

def simulate_crossbar_sds(v):
    v0, v1, v2, v3 = [int(x) for x in v]
    tb = f"""\
`timescale 1ns/1ps
module tb;
    reg  signed [7:0] v0, v1, v2, v3;
    wire signed [9:0] i0, i1, i2, i3;
    crossbar_sds dut (.v0(v0),.v1(v1),.v2(v2),.v3(v3),
                      .i0(i0),.i1(i1),.i2(i2),.i3(i3));
    initial begin
        v0={v0}; v1={v1}; v2={v2}; v3={v3};
        #1;
        $display("i0=%0d i1=%0d i2=%0d i3=%0d",
                 $signed(i0),$signed(i1),$signed(i2),$signed(i3));
        $finish;
    end
endmodule
"""
    with open('tb_crossbar_sds.sv', 'w') as f: f.write(tb)
    r = subprocess.run(['iverilog', '-g2012', '-o', 'sim_crossbar_sds',
                        'tb_crossbar_sds.sv', 'crossbar_sds.sv'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("Compile error:\n", r.stderr); return None
    r = subprocess.run(['vvp', 'sim_crossbar_sds'], capture_output=True, text=True)
    m = re.search(r'i0=(-?\d+) i1=(-?\d+) i2=(-?\d+) i3=(-?\d+)', r.stdout)
    return [int(m.group(k)) for k in range(1, 5)] if m else None

def measure_all_columns_sds(v):
    """SDS analog model: two unipolar sub-buses per column, each with its own R_sense."""
    v = np.array(v, dtype=float)
    results = {}
    for c in range(4):
        pos_rows = np.where(W[:, c] == +1)[0]
        neg_rows = np.where(W[:, c] == -1)[0]
        n_pos, n_neg = len(pos_rows), len(neg_rows)
        I_pos = v[pos_rows].sum() / R / (1 + R_sense * n_pos / R) if n_pos else 0.0
        I_neg = v[neg_rows].sum() / R / (1 + R_sense * n_neg / R) if n_neg else 0.0
        results[c] = {'I': I_pos - I_neg, 'I_pos': I_pos, 'I_neg': I_neg}
    return results

In [12]:
import numpy as np

W = np.array([[1,-1,1,-1],[1,1,-1,-1],[-1,1,1,-1],[-1,-1,-1,1]])
v = np.array([10, 20, 30, 40], dtype=float)
#v = [10, 20, 30, 40]
i_ideal = (W.T @ np.array(v)).tolist()

hw_orig = simulate_crossbar(v)
hw_sds  = simulate_crossbar_sds(v)
print(f"Ideal : {i_ideal}")
print(f"Orig  : {hw_orig}")
print(f"SDS   : {hw_sds}")
assert hw_orig == hw_sds == i_ideal, "Output mismatch!"
print("Digital outputs match.\n")

# Analog model comparison
orig = {c: d['I'] for c, d in {c: {'I': (W[:, c] * np.array(v, dtype=float)).sum() / R /
        (1 + R_sense * np.abs(W[:, c]).sum() / R)} for c in range(4)}.items()}
sds  = measure_all_columns_sds(v)

print(f"{'Col':>4}  {'I_ideal':>9}  {'Orig err':>10}  {'SDS I_pos':>10}  {'SDS I_neg':>10}  {'SDS err':>9}  Floating nodes")
print("-" * 80)
for c in range(4):
    I_orig = orig[c]
    I_sds  = sds[c]['I']
    I_pos  = sds[c]['I_pos']
    I_neg  = sds[c]['I_neg']
    ideal  = i_ideal[c]
    print(f"  {c:>2}  {ideal:>9.4f}  {I_orig-ideal:>10.6f}  {I_pos:>10.4f}  {I_neg:>10.4f}  {I_sds-ideal:>9.6f}  none")

Ideal : [-40.0, 0.0, -20.0, -20.0]
Orig  : [-40, 0, -20, -20]
SDS   : [-40, 0, -20, -20]
Digital outputs match.

 Col    I_ideal    Orig err   SDS I_pos   SDS I_neg    SDS err  Floating nodes
--------------------------------------------------------------------------------
   0   -40.0000    0.159363     29.9401     69.8603   0.079840  none
   1     0.0000    0.000000     49.9002     49.9002   0.000000  none
   2   -20.0000    0.079681     39.9202     59.8802   0.039920  none
   3   -20.0000    0.079681     39.9600     59.8205   0.139502  none
